In [225]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import preprocessor as pp

In [227]:
import importlib
importlib.reload(pp)

<module 'preprocessor' from '/Users/victorli/Desktop/vsc/Programming with Data/DS2500/Battery-RUL-Analysis/preprocessor.py'>

In [229]:
path = 'cleaned_dataset/'

In [231]:
df = pp.read_clean_file(path)
df

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,2010-07-21 15:00:35.093000,4,B0047,0,1,00001.csv,1.674305,NaN,NaN
1,impedance,2010-07-21 16:53:45.968000,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,2010-07-21 17:25:40.670999,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,2010-07-21 20:31:05.000000,24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,2010-07-21 21:02:56.984000,4,B0047,4,5,00005.csv,1.524366,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
7560,impedance,2010-09-30 07:36:45.045999,24,B0055,247,7561,07561.csv,NaN,0.0968087979207628,0.15489738203707232
7561,discharge,2010-09-30 08:08:36.328000,4,B0055,248,7562,07562.csv,1.020138,NaN,NaN
7562,charge,2010-09-30 08:48:54.250000,4,B0055,249,7563,07563.csv,NaN,NaN,NaN
7563,discharge,2010-09-30 11:50:17.687000,4,B0055,250,7564,07564.csv,0.990759,NaN,NaN


In [233]:
BATTERIES = sorted(df['battery_id'].value_counts().index.tolist())
print(len(BATTERIES))
bat_tr, bat_te = train_test_split(BATTERIES, test_size=0.28, random_state=42)
print(bat_tr)
print(bat_te)
df_tr = df[df['battery_id'].isin(bat_tr)]
df_te = df[df['battery_id'].isin(bat_te)]


34
['B0005', 'B0025', 'B0039', 'B0040', 'B0026', 'B0034', 'B0032', 'B0006', 'B0007', 'B0053', 'B0018', 'B0052', 'B0046', 'B0054', 'B0045', 'B0041', 'B0048', 'B0027', 'B0043', 'B0056', 'B0028', 'B0031', 'B0036', 'B0051']
['B0038', 'B0042', 'B0050', 'B0049', 'B0029', 'B0047', 'B0044', 'B0033', 'B0055', 'B0030']


In [235]:
bat_tr, bat_val = train_test_split(BATTERIES,test_size=0.2,random_state=21)
print(bat_tr)
print(bat_val)

df_tr = df[df['battery_id'].isin(bat_tr)]
df_val = df[df['battery_id'].isin(bat_val)]
dis_tr = pp.get_discharges(path, df_tr)
dis_val = pp.get_discharges(path, df_val)

['B0055', 'B0026', 'B0042', 'B0045', 'B0028', 'B0050', 'B0032', 'B0041', 'B0027', 'B0053', 'B0044', 'B0051', 'B0056', 'B0033', 'B0036', 'B0029', 'B0031', 'B0040', 'B0007', 'B0018', 'B0054', 'B0052', 'B0039', 'B0025', 'B0047', 'B0038', 'B0030']
['B0006', 'B0043', 'B0049', 'B0005', 'B0034', 'B0048', 'B0046']


In [236]:
dis_tr.isnull().sum()

start_time             0
ambient_temperature    0
battery_id             0
uid                    0
filename               0
Capacity               0
cycle_number           0
mean_voltage           0
max_voltage            0
min_voltage            0
mean_current           0
max_current            0
mean_temperature       0
max_temperature        0
discharge_time         0
dtype: int64

In [243]:
features = ['ambient_temperature', 
            'cycle_number',
            'mean_voltage',
            'max_voltage',
            'min_voltage',
            'mean_current',
            'max_current',
            'mean_temperature',
            'max_temperature',
            'discharge_time']
disx_tr = dis_tr[features].values
disy_tr = dis_tr['Capacity'].values
disx_val = dis_val[features].values
disy_val = dis_val['Capacity'].values

In [245]:
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    random_state=21
)
rf.fit(disx_tr, disy_tr)

pred_rf = rf.predict(disx_val)

print("RF MAE:", mean_absolute_error(disy_val, pred_rf))
print("RF MSE:", np.sqrt(mean_squared_error(disy_val, pred_rf)))
print("RF R2:", r2_score(disy_val, pred_rf))

RF MAE: 0.054835114621339956
RF MSE: 0.08087236926432277
RF R2: 0.9608273708176315
